# Plotly Dash Dashboard

The runnable app is dashboard.py. The all-sites pie displays the distribution of successful landings across sites. A selected-site pie displays its success/failure ratio. The range slider filters payload scatter points.

Prepared with AI assistance. Data and outputs are reproducible.

In [1]:
"""Reproducible IBM Applied Data Science Capstone analysis.
Run: python analysis.py. Historical course snapshots are included in data/.
"""
from pathlib import Path
import json, sqlite3, io, contextlib, base64
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import folium
from folium.plugins import MarkerCluster, MousePosition
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'; RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':12,'axes.spines.top':False,'axes.spines.right':False})


def saveplot(name):
    plt.tight_layout(); plt.savefig(RESULTS/(name+'.png'),dpi=170,bbox_inches='tight'); plt.close()


def dashboard_evidence():
    df=pd.read_csv(DATA/'spacex_launch_dash.csv'); stats=df.groupby('Launch Site')['class'].agg(['sum','count','mean'])
    fig,ax=plt.subplots(figsize=(8.8,4.4)); ax.pie(stats['sum'],labels=stats.index,autopct='%1.1f%%',colors=['#087f8c','#54a6a6','#e6a04b','#738295']); saveplot('dash_all')
    best=stats['mean'].idxmax(); q=df[df['Launch Site']==best]
    fig,ax=plt.subplots(figsize=(8.8,4.4)); ax.pie([q['class'].sum(),len(q)-q['class'].sum()],labels=['Success','Failure'],autopct='%1.1f%%',colors=['#087f8c','#e6a04b']); saveplot('dash_best')
    for name,lo,hi in [('dash_payload',0,10000),('dash_payload_filtered',2000,6000)]:
        q=df[df['Payload Mass (kg)'].between(lo,hi)]
        fig,ax=plt.subplots(figsize=(8.8,4.4))
        for version,r in q.groupby('Booster Version Category'): ax.scatter(r['Payload Mass (kg)'],r['class'],label=version,alpha=.8)
        ax.set_xlabel('Payload mass (kg)'); ax.set_ylabel('Landing class'); ax.set_yticks([0,1]); ax.legend(fontsize=10); saveplot(name)
    print(stats.to_string()); return {'best_site':best,'stats':stats.reset_index().to_dict('records')}


In [2]:
evidence=dashboard_evidence()

              sum  count      mean
Launch Site                       
CCAFS LC-40     7     26  0.269231
CCAFS SLC-40    3      7  0.428571
KSC LC-39A     10     13  0.769231
VAFB SLC-4E     4     10  0.400000
